In [3]:
"""
使用trl库进行SFT全参微调

"""
from datasets import load_dataset

# 1、处理数据，处理成type为language modeling ，format为对话格式的数据
keyword_data = load_dataset("json",data_files={"train":r"./data/keywords_data_train.jsonl","test":r"./data/keywords_data_test.jsonl"})

In [4]:
# 将数据，转换成带messages，每个message是role 和content的形式
from typing import Dict,List
def data_convert(examples:Dict[str,List]):
    """
    将数据，转换成带messages，每个message是role 和content的形式
    """
    conversation_example_list = examples["conversation"]
    examples_message_list = []
    for example in conversation_example_list:
        message_list = []
        conversation = example[0]
        message_list.append({"role":"user","content":conversation["human"]})
        message_list.append({"role":"assistant","content":conversation["assistant"]})
        examples_message_list.append(message_list)

    return {"messages":examples_message_list}
         

In [5]:
mapped_keyword_data = keyword_data.map(data_convert,batched=True,remove_columns=keyword_data["train"].column_names)

In [6]:
mapped_keyword_data["train"][0]

{'messages': [{'content': '高氟铍矿石在熔炼过程中配入氢氧化铝来脱除其中的氟.结果表明,在配入5％Na2CO3、9.3％Al(OH)3、1400～1500℃熔炼20 min的情况下,BeO回收率达到96％以上,脱氟效果良好(铍玻璃F/BeO能控制在15％以内).为高氟铍矿石的工业应用探索出新的冶炼途径.\n找出上文中的关键词',
   'role': 'user'},
  {'content': '高氟铍矿;配料;熔炼;回收率;脱氟率', 'role': 'assistant'}]}

In [ ]:
# 2、构造SFTConfig实例
import os
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl.trainer.sft_config import SFTConfig
os.environ["TENSORBOARD_LOGGING_DIR"] = "./logs/04_TRL_DEMO"
model = AutoModelForCausalLM.from_pretrained("model/Qwen3-0.6B/")
tokenizer = AutoTokenizer.from_pretrained("model/Qwen3-0.6B/")
training_args = SFTConfig(
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    max_steps=1000,
    num_train_epochs=1,
    logging_strategy="steps",
    logging_steps=100,
    report_to="tensorboard",
    learning_rate=3e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    eval_strategy="steps",
    eval_steps=100,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    load_best_model_at_end=True,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,
    output_dir="finetuned/04_TRL_DEMO",
    bf16=True,
    max_length=710,
    assistant_only_loss=True,
    chat_template_path="./chat_template.jinja"
)

# 3、构造trainer

from trl.trainer.sft_trainer import SFTTrainer

trainer = SFTTrainer(
    args=training_args,
    model=model,
    train_dataset=mapped_keyword_data["train"],
    eval_dataset=mapped_keyword_data["test"],
    processing_class=tokenizer
)

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 1210.40it/s, Materializing param=model.norm.weight]                              
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Truncating eval dataset: 100%|██████████| 500/500 [00:00<00:00, 33161.27 examples/s]


In [12]:
data_loader = trainer.get_train_dataloader()
for batch_data in data_loader:
    # print(batch_data)
    decoded_batch_data = tokenizer.decode(batch_data["input_ids"])
    print(decoded_batch_data[0])
    break

<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 22 June 2026
Reasoning Mode: /think

## Custom Instructions

You are a helpful AI assistant named SmolLM, trained by Hugging Face. Your role as an assistant involves thoroughly exploring questions through a systematic thinking process before providing the final precise and accurate solutions. This requires engaging in a comprehensive cycle of analysis, summarizing, exploration, reassessment, reflection, backtracking, and iteration to develop well-considered thinking process. Please structure your response into two main sections: Thought and Solution using the specified format: <think> Thought section </think> Solution section. In the Thought section, detail your reasoning process in steps. Each step should include detailed considerations such as analysing questions, summarizing relevant findings, brainstorming new ideas, verifying the accuracy of the current steps, refining any errors, and revisiting previous 

In [ ]:
trainer.train()
trainer.save_model("finetuned/04_TRL_DEMO")

In [14]:
# 扩展：获取数据长度的分布情况
def get_lenth_map(example):
    """
    获取数据的长度
    """
    # result: input_ids, attention_mask
    result:Dict = tokenizer.apply_chat_template(example["messages"],tokenize=True)
    return {"length":len(result["input_ids"])}

data_lenth = mapped_keyword_data.map(get_lenth_map,batched=False,remove_columns=["messages"])

Map: 100%|██████████| 500/500 [00:00<00:00, 1426.03 examples/s]


In [18]:
# 得到train当中，第0条样本，经过了chat template处理之后，所得到input_ids的长度，是380
data_lenth["train"][0]["length"]

res_list = data_lenth["train"].to_list()

In [20]:
example_length = [example["length"] for example in res_list]

In [22]:
import pandas as pd
length_series = pd.Series(example_length)

In [31]:
length_series.quantile(0.99)

np.float64(701.0)